# Training Model Sentimen -- Multi-Collection

Notebook ini melatih model klasifikasi sentimen (**Logistic Regression** dan **Naive Bayes**) menggunakan **PySpark MLlib**.
Data diambil dari **dua collection MongoDB**:
- **`comments_sentiment`** (2.000 dokumen dengan label `sentiment`)
- **`comments_preprocessed`** (11.446 dokumen tanpa label)

## Alur Notebook
### Phase 1: Training pada 2000 data berlabel
- Split 80/20 stratified
- Latih Logistic Regression (dengan CrossValidator) dan Naive Bayes
- Evaluasi kedua model, pilih model terbaik berdasarkan F1-macro
- Simpan hasil sebagai **Scenario 1** (`sentiment_training_eval_2000_80_20`)

### Phase 2: Auto-labeling data baru
- Retrain model terbaik pada seluruh 2000 data berlabel
- Prediksi label untuk 11.446 data dari `comments_preprocessed`
- Gabungkan data berlabel + pseudo-labeled

### Phase 3: Skenario tambahan
- **Scenario 2**: Train 2.000, test 11.446 (generalization test)
- **Scenario 3**: 80/20 dari seluruh 13.446 data

## Informasi Data
Fitur: **`text_final`**, Label target: **`sentiment`** (`positif`, `netral`, `negatif`).
Data **imbalanced**: kelas `positif` sangat minoritas (~14,8%).
Strategi mitigasi: class weight pada Logistic Regression + split stratifikasi + evaluasi F1 macro.


## 1. Import Library

In [1]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Iterator

from pymongo import MongoClient
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import (
    CountVectorizer,
    IDF,
    NGram,
    RegexTokenizer,
    StringIndexer,
    VectorAssembler,
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import DataFrame, Row, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print('Library berhasil diimpor.', flush=True)

Library berhasil diimpor.


## 2. Setup Path & Import Modul Lokal

In [2]:
PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    from mongo_comments_loader import (
        create_spark_session as create_project_spark_session,
        load_project_env,
    )
    print('Modul lokal (mongo_comments_loader) berhasil diimpor.')
except ImportError as exc:
    print(f'Error mengimpor modul lokal: {exc}')
    print('Pastikan notebook dijalankan dari root direktori proyek.')

Modul lokal (mongo_comments_loader) berhasil diimpor.


## 3. Konstanta & Konfigurasi

In [3]:
SEED         = 42
TRAIN_RATIO  = 0.80
TEXT_COL     = 'text_final'
LABEL_COL    = 'sentiment'
VALID_LABELS = ('positif', 'netral', 'negatif')

# Collection MongoDB sumber
MONGO_SOURCE_COLLECTION  = 'comments_sentiment'
MONGO_PREPROC_COLLECTION = 'comments_preprocessed'

# Collection MongoDB output
MONGO_EVAL_COLLECTION    = 'sentiment_training_eval'
MONGO_PREDICTION_COLLECTION = 'sentiment_predictions'

print('Konfigurasi:')
print(f'  TEXT_COL          = {TEXT_COL}')
print(f'  LABEL_COL         = {LABEL_COL}')
print(f'  VALID_LABELS      = {VALID_LABELS}')
print(f'  Train/Test ratio  = {TRAIN_RATIO:.0%} / {1-TRAIN_RATIO:.0%}')
print(f'  Source collection = {MONGO_SOURCE_COLLECTION}')
print(f'  Preproc collection= {MONGO_PREPROC_COLLECTION}')
print(f'  Eval collection   = {MONGO_EVAL_COLLECTION}')
print(f'  Pred collection   = {MONGO_PREDICTION_COLLECTION}')


Konfigurasi:
  TEXT_COL          = text_final
  LABEL_COL         = sentiment
  VALID_LABELS      = ('positif', 'netral', 'negatif')
  Train/Test ratio  = 80% / 20%
  Source collection = comments_sentiment
  Preproc collection= comments_preprocessed
  Eval collection   = sentiment_training_eval
  Pred collection   = sentiment_predictions


## 4. Load Environment & Inisialisasi Spark

In [4]:
load_project_env()

MONGO_URI = os.getenv('MONGO_URI', '').strip()
MONGO_DB  = os.getenv('MONGO_DB', 'analisis_sentimen').strip()

if not MONGO_URI:
    raise ValueError('MONGO_URI belum diisi di .env')
if not MONGO_DB:
    raise ValueError('MONGO_DB belum diisi di .env')

print(f'MongoDB DB  : {MONGO_DB}')
print(f'Source col  : {MONGO_SOURCE_COLLECTION}')

spark = create_project_spark_session(
    app_name='sentiment-training-comments-sentiment',
    cores=int(os.getenv('SPARK_CORES', '4')),
    memory=os.getenv('SPARK_MEMORY', '2g'),
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark aktif: master={spark.sparkContext.master}')

MongoDB DB  : analisis_sentimen
Source col  : comments_sentiment
Spark aktif: master=local[20]


## 5. Load & Eksplorasi Data dari MongoDB

In [5]:
def _normalize_value(val):
    """Konversi dict/list ke JSON string agar kompatibel dengan Spark."""
    if isinstance(val, (dict, list)):
        return json.dumps(val, ensure_ascii=False, default=str)
    return val


def load_data(spark: SparkSession) -> tuple:
    """
    Load labeled data dari comments_sentiment dan data unlabeled
    dari comments_preprocessed (tanpa duplikat via comment_id).
    Return: (all_df, labeled_df, unlabeled_df)
    """
    all_docs = []
    seen_ids = set()

    # ---- 1. Load from comments_sentiment (2.000 docs with labels) ----
    with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
        cursor = client[MONGO_DB][MONGO_SOURCE_COLLECTION].find(
            {TEXT_COL: {'$exists': True, '$ne': ''},
             LABEL_COL: {'$exists': True, '$nin': [None, '']}},
            {'_id': 0, 'comment_id': 1, TEXT_COL: 1, LABEL_COL: 1}
        )
        for doc in cursor:
            cid = doc.get('comment_id', '')
            if cid:
                seen_ids.add(cid)
            all_docs.append({
                'comment_id': cid,
                TEXT_COL: _normalize_value(doc[TEXT_COL]),
                LABEL_COL: doc.get(LABEL_COL, ''),
                '_source': 'comments_sentiment',
            })

    n_source = len(all_docs)
    print(f'Dari {MONGO_SOURCE_COLLECTION}: {n_source} dokumen dengan label')

    # ---- 2. Load from comments_preprocessed (tanpa duplikat) ----
    with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
        db = client[MONGO_DB]
        if MONGO_PREPROC_COLLECTION in db.list_collection_names():
            cursor = db[MONGO_PREPROC_COLLECTION].find(
                {TEXT_COL: {'$exists': True, '$ne': ''}},
                {'_id': 0, 'comment_id': 1, TEXT_COL: 1}
            )
            for doc in cursor:
                cid = doc.get('comment_id', '')
                if cid and cid in seen_ids:
                    continue  # skip duplikat
                if cid:
                    seen_ids.add(cid)
                all_docs.append({
                    'comment_id': cid,
                    TEXT_COL: _normalize_value(doc[TEXT_COL]),
                    LABEL_COL: None,
                    '_source': 'comments_preprocessed',
                })
            print(f'Dari {MONGO_PREPROC_COLLECTION}: {len(all_docs) - n_source} dokumen ditambahkan')
        else:
            print(f'Collection {MONGO_PREPROC_COLLECTION} tidak ditemukan.')

    # ---- Create Spark DataFrame ----
    df = spark.createDataFrame(all_docs)
    df = df.select(
        'comment_id',
        F.col(TEXT_COL).cast('string').alias(TEXT_COL),
        F.col(LABEL_COL).cast('string').alias(LABEL_COL),
        '_source',
    )\
        .filter(F.col(TEXT_COL).isNotNull() & (F.trim(F.col(TEXT_COL)) != ''))\
        .cache()

    total_all = df.count()
    labeled = df.filter(F.col(LABEL_COL).isNotNull() & F.col(LABEL_COL).isin(*VALID_LABELS))
    unlabeled = df.filter(F.col(LABEL_COL).isNull() | ~F.col(LABEL_COL).isin(*VALID_LABELS))

    n_labeled = labeled.count()
    n_unlabeled = unlabeled.count()

    print(f'\nTotal: {total_all} dokumen')
    print(f'  Dengan label : {n_labeled}')
    print(f'  Tanpa label  : {n_unlabeled}')

    return df, labeled, unlabeled


# Load data
raw_df, labeled_df, unlabeled_df = load_data(spark)

print(f'Distribusi label pada data berlabel:')
labeled_df.groupBy(LABEL_COL).count().orderBy(LABEL_COL).show(truncate=False)


Dari comments_sentiment: 2000 dokumen dengan label
Dari comments_preprocessed: 11446 dokumen ditambahkan

Total: 13446 dokumen
  Dengan label : 2000
  Tanpa label  : 11446
Distribusi label pada data berlabel:
+---------+-----+
|sentiment|count|
+---------+-----+
|negatif  |1009 |
|netral   |694  |
|positif  |297  |
+---------+-----+



### 5.1 Eksplorasi Distribusi Label & Statistik Teks

In [6]:
print('Distribusi data berdasarkan sumber:')
raw_df.groupBy('_source').count().show(truncate=False)
print()
print('=== Distribusi Label (data berlabel) ===')
labeled_df.groupBy(LABEL_COL).count().orderBy(LABEL_COL).show(truncate=False)
# Persentase
total_labeled = labeled_df.count()
print('Persentase label:')
for row in labeled_df.groupBy(LABEL_COL).count().orderBy(LABEL_COL).collect():
    print(f'  {row[LABEL_COL]:10s}: {row["count"]:5d} ({row["count"]/total_labeled*100:.1f}%)')


Distribusi data berdasarkan sumber:
+---------------------+-----+
|_source              |count|
+---------------------+-----+
|comments_sentiment   |2000 |
|comments_preprocessed|11446|
+---------------------+-----+


=== Distribusi Label (data berlabel) ===
+---------+-----+
|sentiment|count|
+---------+-----+
|negatif  |1009 |
|netral   |694  |
|positif  |297  |
+---------+-----+

Persentase label:
  negatif   :  1009 (50.4%)
  netral    :   694 (34.7%)
  positif   :   297 (14.8%)


## 6. Stratified Split (untuk data berlabel)

Fungsi `stratified_split()` membagi data berlabel menjadi training set dan test set
dengan mempertahankan proporsi label. Parameter `ratio` mengontrol proporsi train (default 0.8).


In [7]:
def stratified_split(df: DataFrame, ratio: float = TRAIN_RATIO, seed: int = SEED):
    """
    Split stratifikasi: menjaga proporsi label di train dan test.
    ratio: proporsi data untuk training (default 0.8).
    """
    window_spec = Window.partitionBy(LABEL_COL).orderBy(F.rand(seed))
    df_ranked = df.withColumn('_rn', F.row_number().over(window_spec))

    label_counts = {r[LABEL_COL]: r['count'] for r in df.groupBy(LABEL_COL).count().collect()}

    train_cond = None
    test_cond  = None
    for lbl in VALID_LABELS:
        cnt = label_counts.get(lbl, 0)
        if cnt == 0:
            continue
        train_limit = int(round(cnt * ratio))
        c_train = (F.col(LABEL_COL) == lbl) & (F.col('_rn') <= train_limit)
        c_test  = (F.col(LABEL_COL) == lbl) & (F.col('_rn') >  train_limit)
        train_cond = c_train if train_cond is None else train_cond | c_train
        test_cond  = c_test  if test_cond  is None else test_cond  | c_test

    train_df = df_ranked.filter(train_cond).drop('_rn').cache()
    test_df  = df_ranked.filter(test_cond).drop('_rn').cache()
    return train_df, test_df


## 7. Definisi Pipeline ML (TF-IDF + Classifier)

In [8]:
def build_pipeline(model_name: str, **kwargs) -> Pipeline:
    tokenizer = RegexTokenizer(
        inputCol=TEXT_COL, outputCol='tokens',
        pattern=r'\s+', gaps=True, minTokenLength=2,
    )
    label_indexer = StringIndexer(
        inputCol=LABEL_COL, outputCol='label_index', handleInvalid='skip'
    )
    ngram = NGram(n=2, inputCol='tokens', outputCol='bigrams')

    name = model_name.lower().strip()

    vocab_uni = kwargs.get('vocab_uni', 8000)
    vocab_bi  = kwargs.get('vocab_bi', 6000)
    min_df    = kwargs.get('min_df', 3.0)

    cv_uni = CountVectorizer(
        inputCol='tokens', outputCol='uni_feat',
        vocabSize=vocab_uni, minDF=min_df, minTF=1,
    )
    cv_bi = CountVectorizer(
        inputCol='bigrams', outputCol='bi_feat',
        vocabSize=vocab_bi, minDF=min_df, minTF=1,
    )

    if name in ('logistic_regression', 'lr', 'logistic regression'):
        assembler = VectorAssembler(
            inputCols=['uni_feat', 'bi_feat'], outputCol='raw_feat'
        )
        idf = IDF(inputCol='raw_feat', outputCol='features', minDocFreq=2)
        clf = LogisticRegression(
            featuresCol='features', labelCol='label_index',
            predictionCol='pred_index',
            maxIter=kwargs.get('max_iter', 300),
            regParam=kwargs.get('reg_param', 0.05),
            elasticNetParam=kwargs.get('elastic_net', 0.15),
            family='multinomial',
            tol=1e-4,
        )
        stages = [tokenizer, ngram, cv_uni, cv_bi, assembler, idf, label_indexer, clf]

    elif name in ('naive_bayes', 'nb', 'naive bayes'):
        use_bigrams = kwargs.get('use_bigrams', False)
        clf = NaiveBayes(
            featuresCol='features', labelCol='label_index',
            predictionCol='pred_index',
            modelType='multinomial',
            smoothing=kwargs.get('smoothing', 0.5),
        )
        if use_bigrams:
            assembler = VectorAssembler(
                inputCols=['uni_feat', 'bi_feat'], outputCol='features'
            )
            stages = [tokenizer, ngram, cv_uni, cv_bi, assembler, label_indexer, clf]
        else:
            assembler = VectorAssembler(
                inputCols=['uni_feat'], outputCol='features'
            )
            stages = [tokenizer, cv_uni, assembler, label_indexer, clf]

    else:
        raise ValueError(f'Model tidak dikenal: {model_name}')

    return Pipeline(stages=stages)


print('Fungsi build_pipeline siap digunakan.')


Fungsi build_pipeline siap digunakan.


## 8. Class Weight (digunakan di train_and_evaluate)

Class weight untuk Logistic Regression dihitung langsung di dalam fungsi `train_and_evaluate()`
menggunakan inverse frequency weighting.


In [9]:
def add_class_weights(df: DataFrame, label_index_col: str = 'label_index') -> DataFrame:
    """
    Tambahkan kolom 'class_weight' menggunakan inverse frequency weighting.
    Formula: weight(c) = total / (n_classes * count(c))
    Ini membantu model memperhatikan kelas minoritas (positif) lebih.
    """
    total = df.count()
    n_classes = len(VALID_LABELS)
    counts = {r[label_index_col]: r['cnt'] for r in
              df.groupBy(label_index_col).agg(F.count('*').alias('cnt')).collect()}

    weight_map = {idx: total / (n_classes * cnt) for idx, cnt in counts.items()}
    print('Class weights (inverse frequency):')
    for idx, w in sorted(weight_map.items()):
        print(f'  label_index={idx:.0f} -> weight={w:.4f}')

    def get_weight(idx):
        return float(weight_map.get(idx, 1.0))

    weight_udf = F.udf(get_weight, 'double')
    return df.withColumn('class_weight', weight_udf(F.col(label_index_col)))


print('Fungsi add_class_weights siap digunakan.')

Fungsi add_class_weights siap digunakan.


## 9. Fungsi Evaluasi Lengkap

In [10]:
def compute_metrics(predictions: DataFrame, labels: list) -> dict:
    """
    Hitung metrik evaluasi lengkap:
      - Accuracy
      - F1 weighted & macro
      - Precision weighted & macro
      - Recall weighted & macro
      - Per-class: precision, recall, f1, support
      - Confusion matrix (sebagai dict)
    """
    total = predictions.count()

    def _eval(metric):
        return MulticlassClassificationEvaluator(
            labelCol='label_index', predictionCol='pred_index', metricName=metric
        ).evaluate(predictions)

    accuracy          = _eval('accuracy')
    f1_weighted       = _eval('f1')
    precision_weighted = _eval('weightedPrecision')
    recall_weighted    = _eval('weightedRecall')

    # Confusion matrix dari groupBy actual vs predicted label
    cm_rows = {
        (r[LABEL_COL], r['pred_label']): r['cnt']
        for r in predictions.groupBy(LABEL_COL, 'pred_label')
                             .agg(F.count('*').alias('cnt'))
                             .collect()
    }

    # Per-class metrics
    per_class = {}
    macro_p = macro_r = macro_f1 = 0.0
    for lbl in VALID_LABELS:
        tp = cm_rows.get((lbl, lbl), 0)
        fp = sum(cm_rows.get((actual, lbl), 0) for actual in VALID_LABELS if actual != lbl)
        fn = sum(cm_rows.get((lbl, pred),   0) for pred   in VALID_LABELS if pred   != lbl)
        support = tp + fn
        p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r  = tp / support    if support   > 0 else 0.0
        f1 = 2*p*r / (p+r)  if (p+r)     > 0 else 0.0
        per_class[lbl] = {'precision': p, 'recall': r, 'f1': f1, 'support': support}
        macro_p  += p
        macro_r  += r
        macro_f1 += f1

    n = len(VALID_LABELS)
    macro_p  /= n
    macro_r  /= n
    macro_f1 /= n

    # Confusion matrix sebagai nested dict
    cm_dict = {
        actual: {pred: cm_rows.get((actual, pred), 0) for pred in VALID_LABELS}
        for actual in VALID_LABELS
    }

    return {
        'total_samples':      total,
        'accuracy':           round(accuracy, 6),
        'f1_weighted':        round(f1_weighted, 6),
        'f1_macro':           round(macro_f1, 6),
        'precision_weighted': round(precision_weighted, 6),
        'precision_macro':    round(macro_p, 6),
        'recall_weighted':    round(recall_weighted, 6),
        'recall_macro':       round(macro_r, 6),
        'per_class':          {k: {m: round(v, 6) if isinstance(v, float) else v
                                   for m, v in cls.items()}
                               for k, cls in per_class.items()},
        'confusion_matrix':   cm_dict,
    }


def print_metrics(metrics: dict, model_name: str, split: str) -> None:
    """Tampilkan ringkasan metrik secara terformat."""
    print(f'\n{"="*60}')
    print(f'  {model_name} | {split}')
    print(f'{"="*60}')
    print(f'  Total sampel      : {metrics["total_samples"]}')
    print(f'  Accuracy          : {metrics["accuracy"]:.4f}')
    print(f'  F1 Weighted       : {metrics["f1_weighted"]:.4f}')
    print(f'  F1 Macro          : {metrics["f1_macro"]:.4f}')
    print(f'  Precision Weighted: {metrics["precision_weighted"]:.4f}')
    print(f'  Precision Macro   : {metrics["precision_macro"]:.4f}')
    print(f'  Recall Weighted   : {metrics["recall_weighted"]:.4f}')
    print(f'  Recall Macro      : {metrics["recall_macro"]:.4f}')
    print()
    print(f'  {"Label":<12} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Support":>10}')
    print(f'  {"-"*56}')
    for lbl, m in metrics['per_class'].items():
        print(f'  {lbl:<12} {m["precision"]:>10.4f} {m["recall"]:>10.4f} {m["f1"]:>10.4f} {m["support"]:>10}')
    print()
    print('  Confusion Matrix (actual \\ predicted):')
    header = f'  {"":12}' + ''.join(f'{p:>12}' for p in VALID_LABELS)
    print(header)
    for actual in VALID_LABELS:
        row = f'  {actual:<12}' + ''.join(f'{metrics["confusion_matrix"][actual].get(p, 0):>12}' for p in VALID_LABELS)
        print(row)


print('Fungsi evaluasi siap.')

Fungsi evaluasi siap.


## 10. Fungsi Simpan ke MongoDB

Dua fungsi penyimpanan:
- `save_eval_to_mongo(record, collection_name)`: simpan metrik evaluasi
- `save_predictions_to_mongo(predictions_df, model_key, collection_name)`: simpan prediksi per-baris


In [11]:
def save_eval_to_mongo(record: dict, collection_name: str = MONGO_EVAL_COLLECTION) -> None:
    """Simpan record hasil evaluasi ke collection tertentu."""
    with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
        client[MONGO_DB][collection_name].insert_one(record)
    print(f'  Disimpan: {MONGO_DB}.{collection_name}')


def save_predictions_to_mongo(predictions_df, model_key: str, collection_name: str) -> None:
    """Simpan (text, actual, predicted) ke collection MongoDB."""
    rows = predictions_df\
        .select('comment_id', TEXT_COL, LABEL_COL, 'pred_label')\
        .collect()
    docs = [
        {
            'model': model_key,
            'comment_id': row['comment_id'],
            TEXT_COL: row[TEXT_COL],
            'actual': row[LABEL_COL],
            'predicted': row['pred_label'],
        }
        for row in rows
    ]
    with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
        client[MONGO_DB][collection_name].insert_many(docs)
    print(f'  Prediksi disimpan: {MONGO_DB}.{collection_name} ({len(docs)} dokumen)')


print('Fungsi penyimpanan MongoDB siap.')


Fungsi penyimpanan MongoDB siap.


## 11. Fungsi Train & Evaluasi Pipeline

In [12]:
def train_and_evaluate(
    model_key,
    model_display,
    train_df,
    test_df,
):

    print(f"\n>>> Training: {model_display} ...", flush=True)

    # Drop kolom tokens jika ada (konflik dengan RegexTokenizer output)
    for col_name in ("tokens",):
        if col_name in train_df.columns:
            train_df = train_df.drop(col_name)
        if col_name in test_df.columns:
            test_df = test_df.drop(col_name)

    pipeline = build_pipeline(model_key)
    stages = pipeline.getStages()

    is_lr = model_key.lower() in ('logistic_regression', 'lr', 'logistic regression')

    if is_lr:
        # === Logistic Regression with CrossValidator ===

        # Step 1: Compute class weights
        label_counts = train_df.groupBy(LABEL_COL).count().collect()
        total = sum(r["count"] for r in label_counts)
        n_class = len(label_counts)

        weight_dict = {
            r[LABEL_COL]: total / (n_class * r["count"])
            for r in label_counts
        }

        print("Class weights (inverse frequency):")
        for k, v in weight_dict.items():
            print(f"  {k} -> {v:.4f}")

        mapping = F.create_map(
            *[x for kv in weight_dict.items() for x in (F.lit(kv[0]), F.lit(float(kv[1])))]
        )
        train_df = train_df.withColumn("class_weight", mapping[F.col(LABEL_COL)])

        # Step 2: Fit feature pipeline (all stages except classifier)
        feat_stages = stages[:-1]
        feat_pipeline = Pipeline(stages=feat_stages)
        feat_model = feat_pipeline.fit(train_df)

        train_feat = feat_model.transform(train_df).cache()
        test_feat = feat_model.transform(test_df).cache()

        # Step 3: CrossValidator for LR
        base_lr = stages[-1]
        base_lr.setWeightCol("class_weight")

        param_grid = ParamGridBuilder() \
            .addGrid(base_lr.regParam, [0.01, 0.05, 0.1]) \
            .addGrid(base_lr.elasticNetParam, [0.0, 0.15, 0.5]) \
            .build()

        evaluator = MulticlassClassificationEvaluator(
            labelCol='label_index', predictionCol='pred_index',
            metricName='f1'
        )
        cv = CrossValidator(
            estimator=base_lr,
            estimatorParamMaps=param_grid,
            evaluator=evaluator,
            numFolds=3,
            seed=SEED,
            parallelism=1,
        )
        cv_model = cv.fit(train_feat)
        best_lr = cv_model.bestModel

        print(f"  Best LR params: regParam={best_lr.getRegParam():.4f}, "
              f"elasticNet={best_lr.getElasticNetParam():.4f}")

        # Step 4: Predict with best model
        train_pred = best_lr.transform(train_feat)
        test_pred = best_lr.transform(test_feat)

        # Step 5: Label mapping from StringIndexerModel
        si_model = feat_model.stages[-1]
        labels = list(si_model.labels)

        when_expr = None
        for i, lbl in enumerate(labels):
            cond = F.col("pred_index").cast("int") == i
            if when_expr is None:
                when_expr = F.when(cond, F.lit(lbl))
            else:
                when_expr = when_expr.when(cond, F.lit(lbl))

        train_pred = train_pred.withColumn("pred_label", when_expr)
        test_pred = test_pred.withColumn("pred_label", when_expr)

        model_obj = cv_model

    else:
        # === Naive Bayes: full-pipeline approach ===
        pipeline = Pipeline(stages=stages)
        fitted = pipeline.fit(train_df)

        si_model = fitted.stages[-2]
        labels = list(si_model.labels)

        when_expr = None
        for i, lbl in enumerate(labels):
            cond = F.col("pred_index").cast("int") == i
            if when_expr is None:
                when_expr = F.when(cond, F.lit(lbl))
            else:
                when_expr = when_expr.when(cond, F.lit(lbl))

        train_pred = fitted.transform(train_df).withColumn("pred_label", when_expr)
        test_pred = fitted.transform(test_df).withColumn("pred_label", when_expr)

        model_obj = fitted

    train_metrics = compute_metrics(train_pred, VALID_LABELS)
    test_metrics = compute_metrics(test_pred, VALID_LABELS)

    print(f"Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"F1 Score : {test_metrics['f1_weighted']:.4f}")
    print(f"Precision: {test_metrics['precision_weighted']:.4f}")
    print(f"Recall   : {test_metrics['recall_weighted']:.4f}")

    return {
        "model": model_obj,
        "train_prediction": train_pred,
        "test_prediction": test_pred,
        "train": train_metrics,
        "test": test_metrics,
    }


## 12. Training 3 Skenario

Melatih dan mengevaluasi model untuk 3 skenario:
1. **80/20 dari 2.000 data berlabel** — baseline
2. **Train 2.000, test 11.446** — generalisasi ke data baru (auto-labeled)
3. **80/20 dari seluruh 13.446 data** — full dataset

Setiap skenario menghasilkan collection evaluasi (prefix `sentiment_training_eval_*`)
dan collection prediksi (prefix `sentiment_predictions_*`).


In [ ]:
import time

MODELS = [
    ('logistic_regression', 'Logistic Regression'),
    ('naive_bayes', 'Naive Bayes'),
]

all_results = {}

# ============================================================
# PHASE 1: Training pada 2000 data berlabel -> Scenario 1
# ============================================================
print('\n' + '='*70)
print('  PHASE 1: Training pada 2000 data berlabel (80/20 stratified)')
print('='*70)

train1, test1 = stratified_split(labeled_df, ratio=TRAIN_RATIO)

phase1_results = {}
best_model_key = None
best_model_score = -1.0
best_model_display = ''

for model_key, model_display in MODELS:
    t0 = time.time()
    results = train_and_evaluate(
        model_key=model_key,
        model_display=model_display,
        train_df=train1,
        test_df=test1,
    )
    elapsed = time.time() - t0
    phase1_results[model_display] = results

    # Simpan evaluasi ke MongoDB (scenario 1)
    record = {
        'model': model_key,
        'model_display': model_display,
        'scenario': 'scenario_1_2000_80_20',
        'scenario_desc': '2000 data berlabel: 80% train / 20% test',
        'timestamp': datetime.now(timezone.utc),
        'train_size': train1.count(),
        'test_size': test1.count(),
        'elapsed_seconds': round(elapsed, 2),
        'best_params': None,
        'train_metrics': results['train'],
        'test_metrics': results['test'],
    }
    if 'logistic' in model_key.lower() and hasattr(results['model'], 'getRegParam'):
        record['best_params'] = {
            'regParam': results['model'].getRegParam(),
            'elasticNetParam': results['model'].getElasticNetParam(),
        }
    save_eval_to_mongo(record, 'sentiment_training_eval_2000_80_20')
    save_predictions_to_mongo(
        results['test_prediction'],
        model_key,
        'sentiment_predictions_2000_80_20',
    )

    # Track best model (by F1-macro on test set)
    f1_macro = results['test']['f1_macro']
    if f1_macro > best_model_score:
        best_model_score = f1_macro
        best_model_key = model_key
        best_model_display = model_display

all_results['scenario_1_2000_80_20'] = phase1_results

print(f'\n>>> Model terbaik: {best_model_display} (F1-macro={best_model_score:.4f})')

# ============================================================
# PHASE 2: Auto-label data menggunakan model terbaik
# ============================================================
n_unlabeled = unlabeled_df.count() if 'unlabeled_df' in dir() else 0

if n_unlabeled > 0 and best_model_key is not None:
    print(f'\n' + '='*70)
    print(f'  PHASE 2: Auto-labeling {n_unlabeled} data dengan model {best_model_display}')
    print('='*70)

    # Retrain best model on ALL 2000 labeled data
    all_train = labeled_df  # full 2000

    pipeline = build_pipeline(best_model_key)
    stages = pipeline.getStages()
    is_lr = 'logistic' in best_model_key.lower()

    if is_lr:
        # Class weights
        lr_counts = all_train.groupBy(LABEL_COL).count().collect()
        lr_total = sum(r['count'] for r in lr_counts)
        lr_n = len(lr_counts)
        wd = {r[LABEL_COL]: lr_total / (lr_n * r['count']) for r in lr_counts}
        mapping = F.create_map(
            *[x for kv in wd.items() for x in (F.lit(kv[0]), F.lit(float(kv[1])))]
        )
        all_train_w = all_train.withColumn('class_weight', mapping[F.col(LABEL_COL)])
        stages[-1].setWeightCol('class_weight')

        feat_stages = stages[:-1]
        feat_model = Pipeline(stages=feat_stages).fit(all_train_w)
        train_feat = feat_model.transform(all_train_w).cache()
        clf_model = stages[-1].fit(train_feat)
        unlabeled_feat = feat_model.transform(unlabeled_df)
        unlabeled_pred = clf_model.transform(unlabeled_feat)
        si_model = feat_model.stages[-1]
    else:
        # NB: full pipeline fit
        fitted = pipeline.fit(all_train)
        unlabeled_pred = fitted.transform(unlabeled_df)
        si_model = fitted.stages[-2]

    # Map prediction indices to label strings
    lbls = list(si_model.labels)
    we = None
    for i, lbl in enumerate(lbls):
        cond = F.col('pred_index').cast('int') == i
        we = F.when(cond, F.lit(lbl)) if we is None else we.when(cond, F.lit(lbl))
    unlabeled_pred = unlabeled_pred.withColumn('pred_label', we)

    pseudo_labeled = unlabeled_pred\
        .select('comment_id', TEXT_COL, F.col('pred_label').alias(LABEL_COL), '_source')\
        .filter(F.col(LABEL_COL).isNotNull())

    n_pseudo = pseudo_labeled.count()
    print(f'  Auto-labeled: {n_pseudo} dokumen')
    all_labeled = labeled_df.unionByName(pseudo_labeled).cache()
    print(f'  Total data berlabel (termasuk auto-label): {all_labeled.count()}')
else:
    print('\nTidak ada data unlabeled atau model terbaik tidak ditemukan.')
    all_labeled = labeled_df.cache()
    n_unlabeled = 0

# ============================================================
# PHASE 3: Skenario 2 & 3
# ============================================================
if n_unlabeled > 0:
    print('\n' + '='*70)
    print('  PHASE 3: Skenario 2 & 3')
    print('='*70)

    # ---- Scenario 2: train=2000, test=11446 (pseudo-labeled) ----
    print('\n>>> Skenario 2: Train 2000 original, test 11446 auto-labeled')
    train2 = labeled_df
    test2 = pseudo_labeled

    try:
        for model_key, model_display in MODELS:
            t0 = time.time()
            results = train_and_evaluate(
                model_key=model_key,
                model_display=model_display,
                train_df=train2,
                test_df=test2,
            )
            elapsed = time.time() - t0

            if 'scenario_2_2000_train_11446_test' not in all_results:
                all_results['scenario_2_2000_train_11446_test'] = {}
            all_results['scenario_2_2000_train_11446_test'][model_display] = results

            record = {
                'model': model_key,
                'model_display': model_display,
                'scenario': 'scenario_2_2000_train_11446_test',
                'scenario_desc': 'Train: 2000 data berlabel | Test: 11446 auto-labeled',
                'timestamp': datetime.now(timezone.utc),
                'train_size': train2.count(),
                'test_size': test2.count(),
                'elapsed_seconds': round(elapsed, 2),
                'best_params': None,
                'train_metrics': results['train'],
                'test_metrics': results['test'],
            }
            if 'logistic' in model_key.lower() and hasattr(results['model'], 'getRegParam'):
                record['best_params'] = {
                    'regParam': results['model'].getRegParam(),
                    'elasticNetParam': results['model'].getElasticNetParam(),
                }
            save_eval_to_mongo(record, 'sentiment_training_eval_2000_train_11446_test')
            save_predictions_to_mongo(
                results['test_prediction'], model_key,
                'sentiment_predictions_2000_train_11446_test',
            )
    except Exception as exc:
        print(f'  Error scenario 2: {exc}', flush=True)
        import traceback; traceback.print_exc()

    # ---- Scenario 3: 80/20 dari seluruh data ----
    print('\n>>> Skenario 3: 80/20 dari seluruh 13446 data')
    train3, test3 = stratified_split(all_labeled, ratio=TRAIN_RATIO)

    try:
        for model_key, model_display in MODELS:
            t0 = time.time()
            results = train_and_evaluate(
                model_key=model_key,
                model_display=model_display,
                train_df=train3,
                test_df=test3,
            )
            elapsed = time.time() - t0

            if 'scenario_3_all_80_20' not in all_results:
                all_results['scenario_3_all_80_20'] = {}
            all_results['scenario_3_all_80_20'][model_display] = results

            record = {
                'model': model_key,
                'model_display': model_display,
                'scenario': 'scenario_3_all_80_20',
                'scenario_desc': '13446 data: 80% train / 20% test',
                'timestamp': datetime.now(timezone.utc),
                'train_size': train3.count(),
                'test_size': test3.count(),
                'elapsed_seconds': round(elapsed, 2),
                'best_params': None,
                'train_metrics': results['train'],
                'test_metrics': results['test'],
            }
            if 'logistic' in model_key.lower() and hasattr(results['model'], 'getRegParam'):
                record['best_params'] = {
                    'regParam': results['model'].getRegParam(),
                    'elasticNetParam': results['model'].getElasticNetParam(),
                }
            save_eval_to_mongo(record, 'sentiment_training_eval_all_80_20')
            save_predictions_to_mongo(
                results['test_prediction'], model_key,
                'sentiment_predictions_all_80_20',
            )
    except Exception as exc:
        print(f'  Error scenario 3: {exc}', flush=True)
        import traceback; traceback.print_exc()
else:
    print('\nTidak ada data unlabeled. Skenario 2 & 3 dilewati.')

print('\n>>> Semua skenario selesai.')



  PHASE 1: Training pada 2000 data berlabel (80/20 stratified)

>>> Training: Logistic Regression ...
Class weights (inverse frequency):
  negatif -> 0.6609
  positif -> 2.2409
  netral -> 0.9610


## 13. Ringkasan Perbandingan Model

In [ ]:
print('\n' + '='*90)
print('  HASIL EVALUASI - SEMUA SKENARIO')
print('='*90)

for scenario_name, scenario_results in all_results.items():
    print(f'\n{"-"*90}')
    print(f'  Skenario: {scenario_name}')
    print(f'{"-"*90}')

    print(f'{"Model":<22} {"Accuracy":>10} {"F1-W":>10} {"F1-M":>10} {"Prec-W":>10} {"Rec-W":>10}')
    print('-' * 76)
    for name, res in scenario_results.items():
        m = res['test']
        print(
            f'{name:<22}'
            f' {m["accuracy"]:>10.4f}'
            f' {m["f1_weighted"]:>10.4f}'
            f' {m["f1_macro"]:>10.4f}'
            f' {m["precision_weighted"]:>10.4f}'
            f' {m["recall_weighted"]:>10.4f}'
        )

    print(f'\n{"Per-Kelas F1":22} {"positif":>12} {"netral":>12} {"negatif":>12}')
    print('-' * 62)
    for name, res in scenario_results.items():
        pc = res['test']['per_class']
        print(
            f'{name:<22}'
            f' {pc.get("positif", {}).get("f1", 0):>12.4f}'
            f' {pc.get("netral",  {}).get("f1", 0):>12.4f}'
            f' {pc.get("negatif", {}).get("f1", 0):>12.4f}'
        )

    # Confusion matrix
    print(f'\n  Confusion Matrix:')
    for name, res in scenario_results.items():
        cm = res['test']['confusion_matrix']
        print(f'  {name}:')
        header = f'    {"":12}' + ''.join(f'{p:>10}' for p in VALID_LABELS)
        print(header)
        for actual in VALID_LABELS:
            row = f'    {actual:<12}' + ''.join(f'{cm[actual].get(p, 0):>10}' for p in VALID_LABELS)
            print(row)


## 14. Stop Spark Session

In [ ]:
spark.stop()
print('Spark session dihentikan.')
print(f'Hasil evaluasi tersimpan di MongoDB collection sentimen_training_eval_*')
print(f'Hasil prediksi tersimpan di MongoDB collection sentiment_predictions_*')
